# Battery knowledge graph demo

Load the 382 battery-cell profiles in `BatteryTypeJson/` into one RDF graph and query it with SPARQL and with the helper functions of the `batterygraph` package.

The profiles use the [EMMO battery domain context](https://w3id.org/emmo/domain/battery/context). It is downloaded once. To work offline, save it locally and set the environment variable `BATTERYGRAPH_CONTEXT` to the file path.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # not needed after `pip install -e .`

from batterygraph import BatteryGraph

bg = BatteryGraph(ROOT / "BatteryTypeJson")
print(f"{len(bg.cells)} cells, {len(bg.manufacturers)} manufacturers, {len(bg):,} triples")

## 1. SPARQL with readable EMMO terms

EMMO terms are opaque IRIs such as `electrochemistry_9b3b4668_…`. In `bg.sparql()` you write them as `{Term}` placeholders (`{hasProperty}`, `{RatedCapacity}`, `{NMC}`), and IRIs in the results are translated back to term names.

In [ ]:
bg.sparql("""
SELECT (COUNT(DISTINCT ?cell) AS ?cells) WHERE {
    ?cell {hasPositiveElectrode}/{hasActiveMaterial} ?m .
    ?m a {NMC} .
}""")

In [ ]:
# Cells with the most linked publications
bg.sparql("""
SELECT ?manufacturer ?name (COUNT(?pub) AS ?publications) WHERE {
    ?cell schema:name ?name ;
          schema:manufacturer/schema:name ?manufacturer ;
          schema:subjectOf ?pub .
} GROUP BY ?manufacturer ?name ORDER BY DESC(?publications) LIMIT 10""")

In [ ]:
# Cells with a rated capacity above 200 Ah and their case type
bg.sparql("""
SELECT ?manufacturer ?name ?capacity ?case WHERE {
    ?cell schema:name ?name ;
          schema:manufacturer/schema:name ?manufacturer ;
          {hasCase} ?c ;
          {hasProperty} ?p .
    ?c a ?case .
    ?p a {RatedCapacity} ; {hasNumericalPart}/{hasNumericalValue} ?capacity .
    FILTER (?capacity > 200)
} ORDER BY DESC(?capacity)""")

## 2. Helper functions

These are the functions the LLM calls in the GraphRAG workflow. Cell names are matched tolerantly (case, spaces and dashes are ignored; the manufacturer may be part of the name).

In [ ]:
bg.query_cell_property("NCR18650BF", "RatedCapacity")

In [ ]:
bg.query_cell_property("CA100", "ChargingCurrent", manufacturer="CALB")

In [ ]:
bg.get_cell_info("Samsung SDI INR21700-50E")

In [ ]:
bg.get_all_cell_properties("LF280K", manufacturer="EVE")